In [ ]:
import os
HOME = os.getcwd()
print(HOME)

/content


## Install dependencies

In [ ]:
%%capture
!pip install -q ultralytics roboflow

## Imports

In [ ]:
from roboflow import Roboflow
from google.colab import userdata
from IPython.display import Image

## Pull dataset

In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets

ROBOFLOW_API_KEY = userdata.get('roboflow_api_key')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(2)
dataset = version.download("yolov8")

mkdir: cannot create directory ‘{HOME}/datasets’: No such file or directory
[Errno 2] No such file or directory: '{HOME}/datasets'
/content
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to football-players-detection-2 in yolov8:: 100%|██████████| 1338/1338 [00:00<00:00, 5636.68it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
mkdir: cannot create directory ‘{HOME}/datasets’: No such file or directory
[Errno 2] No such file or directory: '{HOME}/datasets'
/content
loading Roboflow workspace...
loading Roboflow project...


**NOTE:** Update `data.yaml` structure to align with the latest requirements of the `ultralytics` package.



In [ ]:
!sed -i 's|\(train: \).*|\1../train/images|' {dataset.location}/data.yaml
!sed -i 's|\(val: \).*|\1../valid/images|' {dataset.location}/data.yaml

## Custom training

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolo26s-objv1-150.pt name=yolo26s data={dataset.location}/data.yaml batch=12 epochs=200 patience=30 imgsz=1280 plots=True

[Errno 2] No such file or directory: '{HOME}'
/content
Ultralytics 8.4.212 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/football-players-detection-2/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s-objv1-150.pt, momentum=0.937, mosaic=

In [ ]:
!cp -R '/content/runs' '/content/drive/MyDrive/Hackathon/YOLO_models/players'

## Validate custom model

In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO

video_path = r'/content/drive/MyDrive/Copie de BAR-ATH_tactical.mp4'
cap = cv2.VideoCapture(video_path)
STARTER_FRAME = 11190
cap.set(cv2.CAP_PROP_POS_FRAMES, STARTER_FRAME)
frame_number = STARTER_FRAME
CSV_PATH = '/content/drive/MyDrive/Hackathon/players_tactical_full_match.csv'
rows = []

model = YOLO("/content/runs/detect/yolo26s-4/weights/last.pt")

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    # frame is an image (NumPy array)
    print(f"Frame {frame_number}: {frame.shape}")

    detections = model.predict(source=frame, conf=0.3, show=True, save=True)[0].boxes
    # Extract boxes, confidence scores and class IDs
    for xyxy, score, class_id in zip(
        detections.xyxy,
        detections.conf,
        detections.cls,
    ):
        rows.append({
            "frame": frame_number,
            "x1": float(xyxy[0]),
            "y1": float(xyxy[1]),
            "x2": float(xyxy[2]),
            "y2": float(xyxy[3]),
            "score": float(score),
            "class_id": int(class_id),
        })


    frame_number += 1

    # if frame_number > 90651: # First half only
    # # if frame_number > STARTER_FRAME + 5: # First half only
    #     break

columns = [
    "frame", "x1", "y1", "x2", "y2", "score", "class_id"
]

df = pd.DataFrame(rows, columns=columns)
df.to_csv(CSV_PATH, index=False)

print(df.head())
print(f"Processed {frame_number} frames")
print(f"Saved {len(df)} detections to {CSV_PATH}")

Streaming output truncated to the last 5000 lines.
Frame 198322: (1080, 1920, 3)

0: 736x1280 (no detections), 12.4ms
Speed: 7.1ms preprocess, 12.4ms inference, 0.6ms postprocess per image at shape (1, 3, 736, 1280)
Results saved to /content/runs/detect/predict
Frame 198323: (1080, 1920, 3)

0: 736x1280 (no detections), 12.2ms
Speed: 4.7ms preprocess, 12.2ms inference, 0.6ms postprocess per image at shape (1, 3, 736, 1280)
Results saved to /content/runs/detect/predict
Frame 198324: (1080, 1920, 3)

0: 736x1280 (no detections), 11.7ms
Speed: 4.7ms preprocess, 11.7ms inference, 0.6ms postprocess per image at shape (1, 3, 736, 1280)
Results saved to /content/runs/detect/predict
Frame 198325: (1080, 1920, 3)

0: 736x1280 (no detections), 12.1ms
Speed: 7.0ms preprocess, 12.1ms inference, 0.6ms postprocess per image at shape (1, 3, 736, 1280)
Results saved to /content/runs/detect/predict
Frame 198326: (1080, 1920, 3)

0: 736x1280 (no detections), 11.5ms
Speed: 4.6ms preprocess, 11.5ms infere